# 16 — Final Fast Gradient Difference
## Qwen3.5-2B + Unsloth + LoRA


### Method

**Gradient Difference (GD)** combines two opposing objectives in one update:

`L_GD = L_retain - L_forget`

- `L_retain` is minimised, encouraging useful retained behaviour.
- `L_forget` is subtracted, so optimisation increases loss on forgotten rows.

Each forgotten example is paired with one reproducibly sampled retained-training example.
The update dataset therefore has exactly the same number of pairs as forget rows.

The trajectory is fixed at **five epochs**.
Epoch 5 is the primary model. Validation utility, Truth Ratio, KS and Full Retraining
similarity never select an earlier epoch.


### Final scenarios

| Scenario | Forget rows |
|---|---:|
| Recipient Withdrawal | 426 |
| Invalid Consent | 4,148 |
| Retention Expiry | 6,262 |

These give a small, medium and large deletion request.

## Why this notebook is designed to run quickly

The expensive Qwen base is **not** retrained.

This version uses:

- the same Unsloth Qwen path that successfully completed Full Retraining;
- the already-trained LoRA adapter and binary classification head;
- frozen Qwen base weights;
- physical batch `32 × accumulation 1`, keeping effective batch `32`;
- one-time tokenisation of all 60,000 assessments;
- large inference-only evaluation batches;
- one Qwen load, with exact LoRA/head restoration between scenarios;
- fail-fast NaN/Inf checks;
- epoch-level resume checkpoints;
- one final scenario per cell.

> **Run the three final scenario cells one at a time.**  
> This prevents a later interruption from wasting earlier completed work.

# 1. Environment

The successful Full Retraining run used:

- NVIDIA A100-SXM4-80GB
- PyTorch `2.8.0+cu128`
- CUDA Toolkit `12.8`
- Unsloth `2026.8.22`
- Transformers `5.2.0`
- Triton `3.4.0`

This notebook deliberately does **not** install or upgrade packages.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version as package_version

import gc
import hashlib
import json
import math
import os
import random
import shutil
import time

import numpy as np
import pandas as pd
import torch

from IPython.display import Markdown, display

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Use a CUDA RunPod.")

DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)

print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("GPU:", GPU_NAME)

In [ ]:
# Import Unsloth before Transformers so its Qwen patches are active.

import unsloth
from unsloth import FastVisionModel

from transformers import AutoTokenizer
from peft import PeftModel

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from scipy.stats import ks_2samp

from torch import nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("Unsloth:", package_version("unsloth"))
print("Transformers:", package_version("transformers"))
print("PEFT:", package_version("peft"))
print("Triton:", package_version("triton"))

In [ ]:
environment_checks = {
    "Torch 2.8.0 + CUDA 12.8":
        torch.__version__.startswith("2.8.0+cu128"),
    "Transformers 5.2.0":
        package_version("transformers") == "5.2.0",
    "Unsloth 2026.8.22":
        package_version("unsloth") == "2026.8.22",
    "Triton 3.4.0":
        package_version("triton") == "3.4.0",
}

env_table = pd.DataFrame([
    {"Check": name, "Passed": passed}
    for name, passed in environment_checks.items()
])
env_table["Status"] = env_table["Passed"].map({True: "PASS", False: "FAIL"})
display(env_table[["Check", "Status"]])

if not all(environment_checks.values()):
    raise RuntimeError(
        "Environment does not match the successful Full Retraining stack. "
        "Use the matching fresh RunPod environment rather than upgrading packages here."
    )

if "A100-SXM4-80GB" not in GPU_NAME:
    display(Markdown(
        f"> **Runtime warning:** current GPU is `{GPU_NAME}`. "
        "Saved Full Retraining runtime was measured on A100-SXM4-80GB."
    ))

print("Environment verification: PASS")

# 2. Seed and project paths

Every scenario starts from the same trained baseline and seed `42`.

In [ ]:
SEED = 42

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print("Seed:", SEED)

In [ ]:
def locate_final_submission():
    candidates = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
        Path("/workspace/qub-machine-unlearning"),
    ]

    for candidate in candidates:
        possible = [
            candidate if candidate.name == "final_submission" else None,
            candidate / "final_submission",
            candidate / "code" / "final_submission",
        ]

        for root in possible:
            if root is None:
                continue

            expected = root / "data" / "final" / "kidney_transplant_assessments.csv"
            if expected.is_file():
                return root.resolve()

    raise FileNotFoundError("Could not locate code/final_submission.")

FINAL = locate_final_submission()

MODEL_ROOT = FINAL / "models" / "qwen"
RESULT_ROOT = FINAL / "results" / "qwen"

BASELINE_MODEL = MODEL_ROOT / "baseline"
BASELINE_ADAPTER = BASELINE_MODEL / "adapter"
BASELINE_HEAD = BASELINE_MODEL / "binary_classification_head.pt"
BASELINE_RESULTS = RESULT_ROOT / "original"

FULL_RESULTS = RESULT_ROOT / "full_retraining"

METHOD_MODEL_ROOT = MODEL_ROOT / "gradient_difference"
METHOD_RESULT_ROOT = RESULT_ROOT / "unlearning" / "gradient_difference"

DATA_PATH = FINAL / "data" / "final" / "kidney_transplant_assessments.csv"
FEATURE_PATH = FINAL / "data" / "final" / "classifier_feature_list.json"
SPLIT_PATH = FINAL / "processed_data" / "split_assignments.csv"
MEMBERSHIP_PATH = FINAL / "processed_data" / "deletion_scenario_membership.csv"

required = [
    DATA_PATH,
    FEATURE_PATH,
    SPLIT_PATH,
    MEMBERSHIP_PATH,
    BASELINE_ADAPTER / "adapter_config.json",
    BASELINE_HEAD,
    BASELINE_RESULTS / "experiment_configuration.json",
    BASELINE_RESULTS / "selected_threshold.json",
    BASELINE_RESULTS / "serialisation_specification.json",
    BASELINE_RESULTS / "retained_test_probabilities.csv",
]

missing = [p for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Missing inputs:\n" + "\n".join(map(str, missing)))

print("Final submission:", FINAL)
print("Method output root:", METHOD_RESULT_ROOT)

# 3. Frozen baseline contract

The trained Qwen baseline is reconstructed from:

- the pretrained Qwen3.5 base;
- the saved trained LoRA adapter;
- the saved binary two-class head;
- the saved tokenizer.

Qwen base weights remain frozen throughout unlearning.

In [ ]:
adapter_candidates = [
    BASELINE_ADAPTER / "adapter_model.safetensors",
    BASELINE_ADAPTER / "adapter_model.bin",
]
adapter_weights = [p for p in adapter_candidates if p.is_file()]

if len(adapter_weights) != 1:
    raise RuntimeError(f"Expected one baseline adapter weight file, found {adapter_weights}")

BASELINE_ADAPTER_WEIGHTS = adapter_weights[0]

baseline_config = json.loads(
    (BASELINE_RESULTS / "experiment_configuration.json").read_text(encoding="utf-8")
)
baseline_threshold = json.loads(
    (BASELINE_RESULTS / "selected_threshold.json").read_text(encoding="utf-8")
)
baseline_serialisation = json.loads(
    (BASELINE_RESULTS / "serialisation_specification.json").read_text(encoding="utf-8")
)

MODEL_ID = baseline_config["model_id"]
MAX_SEQ_LENGTH = int(baseline_config["max_seq_length"])
FROZEN_THRESHOLD = float(baseline_threshold["threshold"])

assert baseline_config["run_id"] == "20260829T151430Z"
assert MODEL_ID == "unsloth/Qwen3.5-2B-Base"
assert int(baseline_config["training"]["LoRA rank"]) == 16
assert int(baseline_config["training"]["LoRA alpha"]) == 16
assert float(baseline_config["training"]["LoRA dropout"]) == 0

print("Baseline run:", baseline_config["run_id"])
print("Model:", MODEL_ID)
print("Max sequence length:", MAX_SEQ_LENGTH)
print("Frozen threshold:", FROZEN_THRESHOLD)

# 4. Dataset and deletion membership

In [ ]:
assessments = pd.read_csv(DATA_PATH)
feature_contract = json.loads(FEATURE_PATH.read_text(encoding="utf-8"))
split_assignments = pd.read_csv(SPLIT_PATH)

TARGET = feature_contract["target"]
FEATURES = feature_contract["classifier_features"]

EXPECTED_FEATURES = [
    "recipient_age", "donor_age", "donor_type", "kidney_failure_cause",
    "previous_transplant", "dialysis_months", "abo_compatibility_category",
    "hla_mismatch_count", "antibody_risk_score", "cold_ischaemia_hours",
    "days_since_transplant", "creatinine_mg_dl", "creatinine_change_pct",
    "urine_output_ml_24h", "tacrolimus_level_ng_ml",
    "medication_adherence_pct", "infection_indicator", "previous_rejection",
]

assert len(assessments) == 60_000
assert TARGET == "acute_rejection_within_30_days"
assert FEATURES == EXPECTED_FEATURES

data = assessments.merge(
    split_assignments[["recipient_id", "donor_id", "split"]],
    on=["recipient_id", "donor_id"],
    how="left",
    validate="many_to_one",
)

split_frames = {
    name: data.loc[data["split"].eq(name)].copy().reset_index(drop=True)
    for name in ["train", "validation", "test"]
}

assert {k: len(v) for k, v in split_frames.items()} == {
    "train": 42_024,
    "validation": 8_988,
    "test": 8_988,
}

for frame in split_frames.values():
    frame["assessment_id"] = frame["assessment_id"].astype(str)
    frame["label"] = frame[TARGET].astype("int64")

display(pd.DataFrame([
    {"Split": name, "Rows": len(frame), "Positive rate": frame[TARGET].mean()}
    for name, frame in split_frames.items()
]))

In [ ]:
SCENARIO_LABELS = {
    "recipient_withdrawal": "Recipient Withdrawal",
    "donor_withdrawal": "Donor Withdrawal",
    "hospital_removal": "Hospital Removal",
    "invalid_consent": "Invalid Consent",
    "retention_expiry": "Retention Expiry",
}

EXPECTED_FORGET = {
    "recipient_withdrawal": 426,
    "donor_withdrawal": 1_992,
    "hospital_removal": 4_314,
    "invalid_consent": 4_148,
    "retention_expiry": 6_262,
}

# Full original scenario order is retained for deterministic seeds.
ALL_SCENARIOS = [
    "recipient_withdrawal",
    "donor_withdrawal",
    "invalid_consent",
    "hospital_removal",
    "retention_expiry",
]

FINAL_SCENARIOS = [
    "recipient_withdrawal",
    "invalid_consent",
    "retention_expiry",
]

membership = pd.read_csv(MEMBERSHIP_PATH)
membership["assessment_id"] = membership["assessment_id"].astype(str)

scenario_sets = {}

for scenario, expected_forget in EXPECTED_FORGET.items():
    sm = membership.loc[membership["scenario"].eq(scenario)]

    forget_ids = set(
        sm.loc[sm["membership_type"].eq("training_forget"), "assessment_id"]
    )
    deleted_validation_ids = set(
        sm.loc[sm["membership_type"].eq("deleted_validation"), "assessment_id"]
    )
    deleted_test_ids = set(
        sm.loc[sm["membership_type"].eq("deleted_test"), "assessment_id"]
    )

    training_forget = split_frames["train"].loc[
        split_frames["train"]["assessment_id"].isin(forget_ids)
    ].copy()

    retained_train = split_frames["train"].loc[
        ~split_frames["train"]["assessment_id"].isin(forget_ids)
    ].copy()

    retained_validation = split_frames["validation"].loc[
        ~split_frames["validation"]["assessment_id"].isin(deleted_validation_ids)
    ].copy()

    retained_test = split_frames["test"].loc[
        ~split_frames["test"]["assessment_id"].isin(deleted_test_ids)
    ].copy()

    assert len(training_forget) == expected_forget
    assert len(training_forget) + len(retained_train) == 42_024

    scenario_sets[scenario] = {
        "training_forget": training_forget,
        "retained_train": retained_train,
        "retained_validation": retained_validation,
        "retained_test": retained_test,
    }

display(pd.DataFrame([
    {
        "Scenario": SCENARIO_LABELS[s],
        "Forget train": len(p["training_forget"]),
        "Retain train": len(p["retained_train"]),
        "Retain validation": len(p["retained_validation"]),
        "Retain test": len(p["retained_test"]),
    }
    for s, p in scenario_sets.items()
]))

# 5. Deterministic Qwen text

The same 18 classifier features and the same plain text format are reused.

In [ ]:
FEATURE_LABELS = baseline_serialisation["display_labels"]

BINARY_FEATURES = {
    "previous_transplant",
    "infection_indicator",
    "previous_rejection",
}

assert baseline_serialisation["feature_order"] == FEATURES
assert baseline_serialisation["target_included"] is False
assert baseline_serialisation["identifiers_included"] is False

def format_feature_value(feature, value):
    if pd.isna(value):
        return "missing"
    if feature in BINARY_FEATURES:
        return "yes" if int(value) == 1 else "no"
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.4f}".rstrip("0").rstrip(".")
    return str(value).strip()

def serialize_assessment(row):
    return "\n".join(
        f"{FEATURE_LABELS[feature]}: {format_feature_value(feature, row[feature])}."
        for feature in FEATURES
    )

serialised = data.copy()
serialised["assessment_id"] = serialised["assessment_id"].astype(str)
serialised["text"] = serialised.apply(serialize_assessment, axis=1)

text_by_id = serialised.set_index("assessment_id")["text"]

for parts in scenario_sets.values():
    for frame in parts.values():
        frame["assessment_id"] = frame["assessment_id"].astype(str)
        frame["text"] = frame["assessment_id"].map(text_by_id)
        frame["label"] = frame[TARGET].astype("int64")
        assert frame["text"].notna().all()

print(scenario_sets["recipient_withdrawal"]["retained_train"]["text"].iloc[0])

# 6. Tokenise once

Tokenisation happens once before training, rather than inside every batch of every epoch.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    BASELINE_ADAPTER,
    local_files_only=True,
)

tokenizer.padding_side = "right"

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

TOKEN_CACHE = {}
TOKENISE_BATCH_SIZE = 2_048

all_ids = serialised["assessment_id"].tolist()
all_texts = serialised["text"].tolist()

for start in tqdm(
    range(0, len(all_texts), TOKENISE_BATCH_SIZE),
    desc="Tokenising once",
):
    end = min(start + TOKENISE_BATCH_SIZE, len(all_texts))

    encoded = tokenizer(
        all_texts[start:end],
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
    )

    for assessment_id, token_ids in zip(
        all_ids[start:end],
        encoded["input_ids"],
    ):
        TOKEN_CACHE[assessment_id] = torch.tensor(
            token_ids,
            dtype=torch.int32,
        )

assert len(TOKEN_CACHE) == 60_000

print("Cached token rows:", len(TOKEN_CACHE))

# 7. Cached DataLoader and loss helpers

In [ ]:
class CachedDataset(Dataset):
    def __init__(self, frame):
        frame = frame.reset_index(drop=True)
        self.ids = frame["assessment_id"].astype(str).tolist()
        self.labels = frame["label"].astype(int).tolist()

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        assessment_id = self.ids[index]
        return {
            "assessment_id": assessment_id,
            "input_ids": TOKEN_CACHE[assessment_id],
            "label": self.labels[index],
        }

def cached_collate(rows):
    sequences = [row["input_ids"].long() for row in rows]

    input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=tokenizer.pad_token_id,
    )

    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": torch.tensor([row["label"] for row in rows], dtype=torch.long),
        "assessment_id": [row["assessment_id"] for row in rows],
    }

NUM_WORKERS = 2

def make_loader(frame, batch_size, shuffle=False, generator=None):
    return DataLoader(
        CachedDataset(frame),
        batch_size=batch_size,
        shuffle=shuffle,
        generator=generator,
        collate_fn=cached_collate,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

def positive_weight(frame):
    positives = int(frame["label"].sum())
    negatives = len(frame) - positives
    return float(negatives / positives)

def batch_ce(logits, labels, pos_weight=None):
    if pos_weight is None:
        return F.cross_entropy(logits.float(), labels)

    weights = torch.tensor(
        [1.0, pos_weight],
        dtype=torch.float32,
        device=logits.device,
    )

    return F.cross_entropy(
        logits.float(),
        labels,
        weight=weights,
        reduction="sum",
    ) / len(labels)

# 8. Load the trained baseline once with Unsloth + LoRA

Only the trained LoRA adapter and two-class head are updated.  
The 2.2B Qwen base remains frozen.

In [ ]:
class FP32ClassificationHead(nn.Linear):
    def forward(self, hidden_states):
        return F.linear(hidden_states.float(), self.weight, self.bias)

def clear_device_cache():
    gc.collect()
    torch.cuda.empty_cache()

def load_trained_baseline():
    set_seed()
    clear_device_cache()

    base_model, _processor = FastVisionModel.from_pretrained(
        MODEL_ID,
        load_in_4bit=False,
        load_in_16bit=True,
        max_seq_length=MAX_SEQ_LENGTH,
        use_gradient_checkpointing=False,
    )

    old_head = base_model.get_output_embeddings()

    base_model.set_output_embeddings(
        FP32ClassificationHead(
            old_head.in_features,
            2,
            bias=False,
            device=old_head.weight.device,
            dtype=torch.float32,
        )
    )

    base_model.config.num_labels = 2
    base_model.config.pad_token_id = tokenizer.pad_token_id

    model = PeftModel.from_pretrained(
        base_model,
        BASELINE_ADAPTER,
        is_trainable=True,
    )

    head_state = torch.load(
        BASELINE_HEAD,
        map_location="cpu",
        weights_only=True,
    )

    current = model.state_dict()
    bad_keys = [
        name
        for name, value in head_state.items()
        if name not in current or current[name].shape != value.shape
    ]

    if not head_state or bad_keys:
        raise RuntimeError(f"Binary-head restore failed: {bad_keys[:5]}")

    model.load_state_dict(head_state, strict=False)
    model.config.use_cache = False
    model.to(DEVICE)

    for parameter in model.parameters():
        if parameter.requires_grad:
            parameter.data = parameter.data.float()

    return model

In [ ]:
def capture_trainable(model):
    return {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }

def restore_trainable(model, state):
    named = dict(model.named_parameters())

    with torch.no_grad():
        for name, value in state.items():
            if name not in named:
                raise KeyError(f"Missing trainable parameter: {name}")
            named[name].copy_(value.to(named[name].device))

def state_fingerprint(state):
    digest = hashlib.sha256()

    for name, value in sorted(state.items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(name.encode("utf-8"))
        digest.update(str(tensor.dtype).encode("ascii"))
        digest.update(str(tuple(tensor.shape)).encode("ascii"))
        digest.update(tensor.numpy().tobytes())

    return digest.hexdigest()

In [ ]:
MODEL = load_trained_baseline()

total_parameters = sum(p.numel() for p in MODEL.parameters())
trainable_parameters = sum(
    p.numel() for p in MODEL.parameters() if p.requires_grad
)

display(pd.Series({
    "Total parameters": total_parameters,
    "Trainable parameters": trainable_parameters,
    "Trainable %": 100 * trainable_parameters / total_parameters,
    "GPU": GPU_NAME,
}, name="Value").to_frame())

FROZEN_BASELINE_STATE = capture_trainable(MODEL)
FROZEN_BASELINE_SHA256 = state_fingerprint(FROZEN_BASELINE_STATE)

print("Frozen baseline SHA-256:", FROZEN_BASELINE_SHA256)

# 9. Shared Qwen output and fast evaluation

In [ ]:
def final_token_logits(model, batch):
    input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
    attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

    sequence_logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    ).logits

    final_indices = attention_mask.sum(dim=1) - 1

    logits = sequence_logits[
        torch.arange(input_ids.shape[0], device=DEVICE),
        final_indices,
    ]

    if logits.shape != (input_ids.shape[0], 2):
        raise RuntimeError(f"Unexpected logits shape: {tuple(logits.shape)}")

    return logits

TARGET_EVAL_BATCH_SIZE = 256
MIN_EVAL_BATCH_SIZE = 32

@torch.inference_mode()
def _evaluate_fixed(model, frame, description, batch_size, pos_weight=None):
    model.eval()
    loader = make_loader(frame, batch_size=batch_size, shuffle=False)

    rows = []
    total_loss = 0.0
    total_rows = 0

    for batch in tqdm(loader, desc=description, leave=False):
        y = batch["labels"].to(DEVICE, non_blocking=True)
        logits = final_token_logits(model, batch)
        loss = batch_ce(logits, y, pos_weight)

        probabilities = torch.softmax(logits.float(), dim=1)[:, 1]

        n = len(y)
        total_loss += float(loss.item()) * n
        total_rows += n

        rows.extend(
            {
                "assessment_id": assessment_id,
                "label": int(label),
                "probability_class_1": float(probability),
            }
            for assessment_id, label, probability in zip(
                batch["assessment_id"],
                batch["labels"].tolist(),
                probabilities.cpu().tolist(),
            )
        )

    return pd.DataFrame(rows), total_loss / total_rows

def evaluate_frame(model, frame, description, pos_weight=None):
    batch_size = TARGET_EVAL_BATCH_SIZE

    while batch_size >= MIN_EVAL_BATCH_SIZE:
        try:
            predictions, loss = _evaluate_fixed(
                model,
                frame,
                description,
                batch_size,
                pos_weight,
            )
            return predictions, loss, batch_size

        except torch.cuda.OutOfMemoryError:
            clear_device_cache()
            batch_size //= 2
            print("Retrying evaluation at batch:", batch_size)

    raise RuntimeError("Evaluation did not fit on the GPU.")

In [ ]:
def utility_metrics(predictions):
    y = predictions["label"].to_numpy()
    p = predictions["probability_class_1"].to_numpy()
    pred = (p >= FROZEN_THRESHOLD).astype(int)

    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()

    return {
        "n": len(y),
        "pr_auc": average_precision_score(y, p),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "binary_cross_entropy": log_loss(y, p, labels=[0, 1]),
        "f1": f1_score(y, pred, zero_division=0),
        "auroc": roc_auc_score(y, p),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "specificity": tn / (tn + fp),
        "threshold": FROZEN_THRESHOLD,
    }

def truth_components(labels, probabilities, epsilon=1e-12):
    p_true = np.where(labels == 1, probabilities, 1 - probabilities)
    p_incorrect = 1 - p_true
    truth_ratio = (p_incorrect + epsilon) / (p_true + epsilon)
    return p_true, p_incorrect, truth_ratio

def forgetting_evidence(approximate, full_retraining):
    full_retraining = full_retraining.copy()
    full_retraining["assessment_id"] = full_retraining["assessment_id"].astype(str)

    joined = approximate.merge(
        full_retraining,
        on="assessment_id",
        suffixes=("_approximate", "_full"),
        validate="one_to_one",
    )

    assert len(joined) == len(approximate) == len(full_retraining)
    assert np.array_equal(joined["label_approximate"], joined["label_full"])

    y = joined["label_approximate"].to_numpy()
    ap = joined["probability_class_1_approximate"].to_numpy()
    fp = joined["probability_class_1_full"].to_numpy()

    at, ai, ar = truth_components(y, ap)
    ft, fi, fr = truth_components(y, fp)

    ks = ks_2samp(ar, fr, alternative="two-sided", method="auto")

    values = pd.DataFrame({
        "assessment_id": joined["assessment_id"],
        "label": y,
        "approximate_probability": ap,
        "full_retraining_probability": fp,
        "approximate_p_true": at,
        "approximate_p_incorrect": ai,
        "approximate_truth_ratio": ar,
        "full_p_true": ft,
        "full_p_incorrect": fi,
        "full_truth_ratio": fr,
        "epsilon": 1e-12,
    })

    return values, {
        "forget_rows": len(values),
        "ks_statistic": float(ks.statistic),
        "ks_p_value": float(ks.pvalue),
    }

# 10. Full Retraining references and method output paths

In [ ]:
def full_reference_paths(scenario):
    folder = FULL_RESULTS / scenario
    return {
        "complete": folder / "COMPLETE.json",
        "forget": folder / "forget_set_probabilities.csv",
    }

def load_full_reference(scenario):
    paths = full_reference_paths(scenario)

    missing = [p for p in paths.values() if not p.is_file()]
    if missing:
        raise FileNotFoundError(
            "Missing Full Retraining reference:\n" + "\n".join(map(str, missing))
        )

    complete = json.loads(paths["complete"].read_text(encoding="utf-8"))

    forget = pd.read_csv(paths["forget"])
    forget["assessment_id"] = forget["assessment_id"].astype(str)

    return complete, forget

def scenario_paths(scenario):
    model_dir = METHOD_MODEL_ROOT / scenario
    result_dir = METHOD_RESULT_ROOT / scenario

    return {
        "model_dir": model_dir,
        "adapter": model_dir / "adapter",
        "head": model_dir / "binary_classification_head.pt",
        "result_dir": result_dir,
        "checkpoint": result_dir / "_resume_checkpoint.pt",
        "complete": result_dir / "COMPLETE.json",
    }

def prepare_scenario(scenario):
    paths = scenario_paths(scenario)

    if paths["complete"].is_file():
        payload = json.loads(paths["complete"].read_text(encoding="utf-8"))
        if payload.get("status") == "complete":
            return "skip"

    if paths["checkpoint"].is_file():
        return "resume"

    if paths["model_dir"].exists() or paths["result_dir"].exists():
        stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        archive = METHOD_RESULT_ROOT / "_partial_archive" / stamp / scenario
        archive.mkdir(parents=True, exist_ok=True)

        for source in [paths["model_dir"], paths["result_dir"]]:
            if source.exists():
                shutil.move(str(source), str(archive / source.name))

        print("Archived previous partial run:", archive)

    paths["model_dir"].mkdir(parents=True, exist_ok=True)
    paths["result_dir"].mkdir(parents=True, exist_ok=True)

    return "run"

In [ ]:
def save_selected_model(model, scenario):
    paths = scenario_paths(scenario)
    paths["adapter"].mkdir(parents=True, exist_ok=True)

    model.save_pretrained(paths["adapter"], safe_serialization=True)
    tokenizer.save_pretrained(paths["adapter"])

    head_state = {
        name: value.detach().cpu()
        for name, value in model.state_dict().items()
        if "lm_head" in name
    }

    if not head_state:
        raise RuntimeError("Binary head not found while saving.")

    torch.save(head_state, paths["head"])

# 11. Gradient Difference configuration

The final GD contract is preserved:

| Setting | Value |
|---|---:|
| AdamW learning rate | `1e-5` |
| Weight decay | `0` |
| Effective pair batch | `32` |
| Epochs | `5` |
| Gradient clip | `1.0` |
| Forget weight λ | `1.0` |
| Selection | fixed epoch 5 |

For speed, each physical pair batch contains 32 forget rows and 32 retained partners.

Forget and retain examples are concatenated into **one Qwen forward pass** per pair batch,
then split before the two losses are calculated. This preserves the objective while avoiding
two separate model calls.

In [ ]:
GD_LEARNING_RATE = 1e-5
GD_WEIGHT_DECAY = 0.0

GD_PAIR_BATCH_SIZE = 32
GD_GRADIENT_ACCUMULATION = 1
GD_EFFECTIVE_PAIR_BATCH = (
    GD_PAIR_BATCH_SIZE * GD_GRADIENT_ACCUMULATION
)

GD_EPOCHS = 5
GD_GRADIENT_CLIP_NORM = 1.0
GD_FORGET_WEIGHT_LAMBDA = 1.0

assert GD_EFFECTIVE_PAIR_BATCH == 32

display(pd.Series({
    "Learning rate": GD_LEARNING_RATE,
    "Weight decay": GD_WEIGHT_DECAY,
    "Physical pair batch": GD_PAIR_BATCH_SIZE,
    "Gradient accumulation": GD_GRADIENT_ACCUMULATION,
    "Effective pair batch": GD_EFFECTIVE_PAIR_BATCH,
    "Epochs": GD_EPOCHS,
    "Gradient clip": GD_GRADIENT_CLIP_NORM,
    "Forget lambda": GD_FORGET_WEIGHT_LAMBDA,
    "Selection": "fixed epoch 5",
}, name="GD configuration").to_frame())

# 12. Deterministic forget-retain pairing

Every epoch samples one retained-training row for every forget row.

The same Qwen final-experiment seed rule is preserved:

`42 + 20,000 × (scenario index + 1) + epoch`

In [ ]:
def paired_retain_indices(scenario, epoch):
    scenario_index = ALL_SCENARIOS.index(scenario)

    sample_seed = (
        SEED
        + 20_000 * (scenario_index + 1)
        + epoch
    )

    rng = np.random.default_rng(sample_seed)

    return rng.integers(
        0,
        len(scenario_sets[scenario]["retained_train"]),
        size=len(scenario_sets[scenario]["training_forget"]),
    )

# Show the deterministic sampling audit without training.
audit_rows = []

for scenario in FINAL_SCENARIOS:
    for epoch in range(1, GD_EPOCHS + 1):
        indices = paired_retain_indices(scenario, epoch)

        audit_rows.append({
            "Scenario": SCENARIO_LABELS[scenario],
            "Epoch": epoch,
            "Forget examples": len(indices),
            "Retain examples sampled": len(indices),
            "Unique retained partners": len(np.unique(indices)),
        })

display(pd.DataFrame(audit_rows))

# 13. Paired cached DataLoader

The paired loader uses cached tokens for both sides.

It returns one combined batch so Qwen is called once rather than twice.

In [ ]:
class PairedCachedDataset(Dataset):
    def __init__(self, forget_frame, retain_frame, retain_indices):
        self.forget = forget_frame.reset_index(drop=True)
        self.retain = retain_frame.reset_index(drop=True)
        self.retain_indices = np.asarray(retain_indices, dtype=np.int64)

        assert len(self.forget) == len(self.retain_indices)

    def __len__(self):
        return len(self.forget)

    def __getitem__(self, index):
        forget_row = self.forget.iloc[index]
        retain_row = self.retain.iloc[int(self.retain_indices[index])]

        return {
            "forget_id": str(forget_row["assessment_id"]),
            "forget_label": int(forget_row["label"]),
            "retain_id": str(retain_row["assessment_id"]),
            "retain_label": int(retain_row["label"]),
        }

def paired_cached_collate(rows):
    forget_ids = [row["forget_id"] for row in rows]
    retain_ids = [row["retain_id"] for row in rows]

    # Forget first, retained second.
    all_ids = forget_ids + retain_ids

    sequences = [
        TOKEN_CACHE[assessment_id].long()
        for assessment_id in all_ids
    ]

    input_ids = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=tokenizer.pad_token_id,
    )

    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    forget_labels = torch.tensor(
        [row["forget_label"] for row in rows],
        dtype=torch.long,
    )

    retain_labels = torch.tensor(
        [row["retain_label"] for row in rows],
        dtype=torch.long,
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "forget_labels": forget_labels,
        "retain_labels": retain_labels,
        "pair_count": len(rows),
    }

def make_paired_loader(scenario, epoch):
    parts = scenario_sets[scenario]

    dataset = PairedCachedDataset(
        parts["training_forget"],
        parts["retained_train"],
        paired_retain_indices(scenario, epoch),
    )

    return DataLoader(
        dataset,
        batch_size=GD_PAIR_BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED + epoch),
        collate_fn=paired_cached_collate,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
    )

In [ ]:
def paired_logits(model, batch):
    input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
    attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)

    sequence_logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    ).logits

    final_indices = attention_mask.sum(dim=1) - 1

    logits = sequence_logits[
        torch.arange(input_ids.shape[0], device=DEVICE),
        final_indices,
    ]

    pair_count = int(batch["pair_count"])

    if logits.shape != (2 * pair_count, 2):
        raise RuntimeError(f"Unexpected paired logits shape: {tuple(logits.shape)}")

    forget_logits = logits[:pair_count]
    retain_logits = logits[pair_count:]

    return forget_logits, retain_logits

# 14. GD smoke test

One paired batch confirms:

- the objective has the correct sign;
- loss and gradients are finite;
- the baseline is restored exactly afterwards.

In [ ]:
def run_gd_smoke_test():
    scenario = "recipient_withdrawal"

    restore_trainable(MODEL, FROZEN_BASELINE_STATE)

    parameters = [p for p in MODEL.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        parameters,
        lr=GD_LEARNING_RATE,
        weight_decay=GD_WEIGHT_DECAY,
    )

    loader = make_paired_loader(scenario, epoch=1)
    batch = next(iter(loader))

    forget_labels = batch["forget_labels"].to(DEVICE, non_blocking=True)
    retain_labels = batch["retain_labels"].to(DEVICE, non_blocking=True)

    forget_logits, retain_logits = paired_logits(MODEL, batch)

    forget_loss = batch_ce(forget_logits, forget_labels)
    retain_loss = batch_ce(retain_logits, retain_labels)

    objective = (
        retain_loss
        - GD_FORGET_WEIGHT_LAMBDA * forget_loss
    )

    if not bool(torch.isfinite(objective).item()):
        raise RuntimeError("GD smoke objective is non-finite.")

    objective.backward()

    grad_norm = torch.nn.utils.clip_grad_norm_(
        parameters,
        GD_GRADIENT_CLIP_NORM,
    )

    if not bool(torch.isfinite(grad_norm).item()):
        raise RuntimeError("GD smoke gradients are non-finite.")

    optimizer.step()

    report = {
        "Forget CE": float(forget_loss.item()),
        "Retain CE": float(retain_loss.item()),
        "Combined objective": float(objective.item()),
        "Independent retain - forget": float(
            (retain_loss - forget_loss).item()
        ),
        "Gradient norm": float(grad_norm.item()),
    }

    assert np.isclose(
        report["Combined objective"],
        report["Independent retain - forget"],
        atol=1e-7,
        rtol=0,
    )

    restore_trainable(MODEL, FROZEN_BASELINE_STATE)
    MODEL.zero_grad(set_to_none=True)

    assert state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256

    del optimizer
    clear_device_cache()

    display(pd.Series(report, name="GD smoke test").to_frame())
    print("GD SMOKE TEST: PASS")

run_gd_smoke_test()

# 15. Fixed five-epoch Gradient Difference training

No validation checkpoint selection occurs.

Epoch 5 is always the primary result unless the run fails numerically.

In [ ]:
def save_gd_checkpoint(
    path,
    *,
    scenario,
    epoch,
    current_state,
    history,
    optimizer,
    elapsed_seconds,
):
    torch.save(
        {
            "scenario": scenario,
            "epoch": epoch,
            "current_state": current_state,
            "history": history,
            "optimizer_state": optimizer.state_dict(),
            "elapsed_seconds": elapsed_seconds,
            "baseline_sha256": FROZEN_BASELINE_SHA256,
        },
        path,
    )

def optimizer_to_device(optimizer):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(DEVICE)

In [ ]:
def train_gd(model, scenario, resume=False):
    paths = scenario_paths(scenario)

    restore_trainable(model, FROZEN_BASELINE_STATE)

    parameters = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        parameters,
        lr=GD_LEARNING_RATE,
        weight_decay=GD_WEIGHT_DECAY,
    )

    start_epoch = 1
    history = []
    previous_seconds = 0.0

    if resume:
        checkpoint = torch.load(
            paths["checkpoint"],
            map_location="cpu",
            weights_only=False,
        )

        if checkpoint["baseline_sha256"] != FROZEN_BASELINE_SHA256:
            raise RuntimeError("GD checkpoint belongs to another baseline.")

        restore_trainable(model, checkpoint["current_state"])

        optimizer.load_state_dict(checkpoint["optimizer_state"])
        optimizer_to_device(optimizer)

        start_epoch = int(checkpoint["epoch"]) + 1
        history = checkpoint["history"]
        previous_seconds = checkpoint["elapsed_seconds"]

        print("Resuming GD from epoch:", start_epoch)

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()

    for epoch in range(start_epoch, GD_EPOCHS + 1):
        loader = make_paired_loader(scenario, epoch)

        model.train()
        optimizer.zero_grad(set_to_none=True)

        forget_total = torch.zeros((), dtype=torch.float64, device=DEVICE)
        retain_total = torch.zeros((), dtype=torch.float64, device=DEVICE)
        combined_total = torch.zeros((), dtype=torch.float64, device=DEVICE)
        rows_seen = 0

        for step, batch in enumerate(
            tqdm(loader, desc=f"GD {scenario} epoch {epoch}"),
            1,
        ):
            forget_labels = batch["forget_labels"].to(
                DEVICE,
                non_blocking=True,
            )
            retain_labels = batch["retain_labels"].to(
                DEVICE,
                non_blocking=True,
            )

            forget_logits, retain_logits = paired_logits(model, batch)

            forget_loss = batch_ce(forget_logits, forget_labels)
            retain_loss = batch_ce(retain_logits, retain_labels)

            objective = (
                retain_loss
                - GD_FORGET_WEIGHT_LAMBDA * forget_loss
            )

            if step == 1 or step % 25 == 0:
                if not bool(torch.isfinite(objective).item()):
                    raise RuntimeError(
                        f"Non-finite GD objective: {scenario}, epoch {epoch}, step {step}"
                    )

            (objective / GD_GRADIENT_ACCUMULATION).backward()

            pair_count = len(forget_labels)

            forget_total += forget_loss.detach().double() * pair_count
            retain_total += retain_loss.detach().double() * pair_count
            combined_total += objective.detach().double() * pair_count
            rows_seen += pair_count

            update_now = (
                step % GD_GRADIENT_ACCUMULATION == 0
                or step == len(loader)
            )

            if update_now:
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    parameters,
                    GD_GRADIENT_CLIP_NORM,
                )

                if not bool(torch.isfinite(grad_norm).item()):
                    raise RuntimeError(
                        f"Non-finite GD gradients: {scenario}, epoch {epoch}, step {step}"
                    )

                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

        row = {
            "scenario": scenario,
            "epoch": epoch,
            "mean_forget_ce": float((forget_total / rows_seen).item()),
            "mean_retain_ce": float((retain_total / rows_seen).item()),
            "mean_combined_objective": float((combined_total / rows_seen).item()),
            "forget_examples": rows_seen,
            "sampled_retain_examples": rows_seen,
        }

        history.append(row)
        display(pd.DataFrame([row]).round(6))

        elapsed = previous_seconds + (time.perf_counter() - started)

        save_gd_checkpoint(
            paths["checkpoint"],
            scenario=scenario,
            epoch=epoch,
            current_state=capture_trainable(model),
            history=history,
            optimizer=optimizer,
            elapsed_seconds=elapsed,
        )

    torch.cuda.synchronize()

    training_seconds = previous_seconds + (time.perf_counter() - started)

    if not history or int(history[-1]["epoch"]) != GD_EPOCHS:
        raise RuntimeError("GD did not reach the required fixed epoch 5.")

    summary = {
        "scenario": scenario,
        "selected_epoch": GD_EPOCHS,
        "epochs_executed": GD_EPOCHS,
        "training_seconds": float(training_seconds),
        "peak_gpu_memory_gib": float(
            torch.cuda.max_memory_allocated() / 1024**3
        ),
        "learning_rate": GD_LEARNING_RATE,
        "weight_decay": GD_WEIGHT_DECAY,
        "physical_pair_batch_size": GD_PAIR_BATCH_SIZE,
        "gradient_accumulation": GD_GRADIENT_ACCUMULATION,
        "effective_pair_batch_size": GD_EFFECTIVE_PAIR_BATCH,
        "epochs": GD_EPOCHS,
        "gradient_clip_norm": GD_GRADIENT_CLIP_NORM,
        "forget_weight_lambda": GD_FORGET_WEIGHT_LAMBDA,
        "selection_rule": "fixed epoch 5; no validation/test/KS/full-retraining model selection",
        "objective": "retain CE - forget CE",
        "paired_sampling": "one deterministic retained-training partner per forget row per epoch",
        "implementation": "Unsloth 16-bit LoRA gradient difference; combined forget+retain forward pass",
    }

    return history, summary

# 16. Final GD evaluation and save

In [ ]:
def evaluate_and_save_gd(model, scenario, history, summary):
    paths = scenario_paths(scenario)
    parts = scenario_sets[scenario]

    retained_predictions, _, retained_batch = evaluate_frame(
        model,
        parts["retained_test"],
        f"GD retained test {scenario}",
    )

    forget_predictions, _, forget_batch = evaluate_frame(
        model,
        parts["training_forget"],
        f"GD forget set {scenario}",
    )

    metrics = utility_metrics(retained_predictions)

    full_complete, full_forget = load_full_reference(scenario)

    truth_values, forgetting = forgetting_evidence(
        forget_predictions,
        full_forget,
    )

    save_selected_model(model, scenario)

    pd.DataFrame(history).to_csv(
        paths["result_dir"] / "training_history.csv",
        index=False,
    )
    pd.DataFrame([metrics]).to_csv(
        paths["result_dir"] / "retained_test_metrics.csv",
        index=False,
    )
    retained_predictions.to_csv(
        paths["result_dir"] / "retained_test_probabilities.csv",
        index=False,
    )
    forget_predictions.to_csv(
        paths["result_dir"] / "forget_set_probabilities.csv",
        index=False,
    )
    truth_values.to_csv(
        paths["result_dir"] / "truth_ratio_values.csv",
        index=False,
    )
    pd.DataFrame([forgetting]).to_csv(
        paths["result_dir"] / "forgetting_metrics.csv",
        index=False,
    )

    full_seconds = float(full_complete["training_seconds"])
    speed_up = full_seconds / summary["training_seconds"]

    complete = {
        "status": "complete",
        **summary,
        **forgetting,
        "retained_test_rows": len(retained_predictions),
        "training_forget_rows": len(forget_predictions),
        "pr_auc": metrics["pr_auc"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "binary_cross_entropy": metrics["binary_cross_entropy"],
        "f1": metrics["f1"],
        "auroc": metrics["auroc"],
        "full_retraining_training_seconds": full_seconds,
        "speed_up": float(speed_up),
        "current_gpu": GPU_NAME,
        "retained_test_evaluation_batch": retained_batch,
        "forget_evaluation_batch": forget_batch,
    }

    (paths["result_dir"] / "configuration.json").write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )

    paths["complete"].write_text(
        json.dumps(complete, indent=2),
        encoding="utf-8",
    )

    if paths["checkpoint"].is_file():
        paths["checkpoint"].unlink()

    print(f"COMPLETE: {SCENARIO_LABELS[scenario]}")
    print("Primary epoch:", summary["selected_epoch"])
    print(f"Training minutes: {summary['training_seconds'] / 60:.2f}")
    print(f"Speed-up: {speed_up:.2f}x")

    return complete

In [ ]:
def run_gd_scenario(scenario):
    if scenario not in FINAL_SCENARIOS:
        raise ValueError(f"{scenario} is not a final scenario.")

    action = prepare_scenario(scenario)

    if action == "skip":
        print("Already complete:", SCENARIO_LABELS[scenario])
        return json.loads(
            scenario_paths(scenario)["complete"].read_text(encoding="utf-8")
        )

    restore_trainable(MODEL, FROZEN_BASELINE_STATE)

    try:
        history, summary = train_gd(
            MODEL,
            scenario,
            resume=(action == "resume"),
        )

        return evaluate_and_save_gd(
            MODEL,
            scenario,
            history,
            summary,
        )

    finally:
        restore_trainable(MODEL, FROZEN_BASELINE_STATE)
        MODEL.zero_grad(set_to_none=True)
        clear_device_cache()

        assert (
            state_fingerprint(capture_trainable(MODEL))
            == FROZEN_BASELINE_SHA256
        )

# 17. Final preflight

In [ ]:
checks = []

def check(name, passed):
    checks.append({"Check": name, "Passed": bool(passed)})

check("Exact environment", all(environment_checks.values()))
check(
    "Current model equals frozen baseline",
    state_fingerprint(capture_trainable(MODEL)) == FROZEN_BASELINE_SHA256,
)
check("GD effective pair batch = 32", GD_EFFECTIVE_PAIR_BATCH == 32)
check("GD learning rate = 1e-5", GD_LEARNING_RATE == 1e-5)
check("GD weight decay = 0", GD_WEIGHT_DECAY == 0)
check("GD fixed epochs = 5", GD_EPOCHS == 5)
check("GD gradient clip = 1.0", GD_GRADIENT_CLIP_NORM == 1.0)
check("GD lambda = 1.0", GD_FORGET_WEIGHT_LAMBDA == 1.0)

for scenario in FINAL_SCENARIOS:
    check(
        f"{SCENARIO_LABELS[scenario]} forget count",
        len(scenario_sets[scenario]["training_forget"]) == EXPECTED_FORGET[scenario],
    )

    paths = full_reference_paths(scenario)
    check(
        f"{SCENARIO_LABELS[scenario]} Full Retraining COMPLETE",
        paths["complete"].is_file(),
    )
    check(
        f"{SCENARIO_LABELS[scenario]} Full Retraining forget probabilities",
        paths["forget"].is_file(),
    )

preflight = pd.DataFrame(checks)
preflight["Status"] = preflight["Passed"].map({True: "PASS", False: "FAIL"})
display(preflight[["Check", "Status"]])

if not preflight["Passed"].all():
    raise RuntimeError("GD PRE-FLIGHT FAILED. Training has not started.")

print("GD PRE-FLIGHT: PASS")

# Final GD run 1 — Recipient Withdrawal

Forget rows: **426**

In [ ]:
recipient_withdrawal_gd_result = run_gd_scenario("recipient_withdrawal")
display(pd.Series(recipient_withdrawal_gd_result, name="Recipient Withdrawal").to_frame())

# Final GD run 2 — Invalid Consent

Forget rows: **4,148**

In [ ]:
invalid_consent_gd_result = run_gd_scenario("invalid_consent")
display(pd.Series(invalid_consent_gd_result, name="Invalid Consent").to_frame())

# Final GD run 3 — Retention Expiry

Forget rows: **6,262**

In [ ]:
retention_expiry_gd_result = run_gd_scenario("retention_expiry")
display(pd.Series(retention_expiry_gd_result, name="Retention Expiry").to_frame())

# Final saved GD table

In [ ]:
rows = []

for scenario in FINAL_SCENARIOS:
    paths = scenario_paths(scenario)

    if not paths["complete"].is_file():
        continue

    complete = json.loads(paths["complete"].read_text(encoding="utf-8"))
    metrics = pd.read_csv(
        paths["result_dir"] / "retained_test_metrics.csv"
    ).iloc[0]

    rows.append({
        "Scenario": SCENARIO_LABELS[scenario],
        "Forget rows": complete["training_forget_rows"],
        "Primary epoch": complete["selected_epoch"],
        "PR-AUC": metrics["pr_auc"],
        "Balanced Accuracy": metrics["balanced_accuracy"],
        "BCE": metrics["binary_cross_entropy"],
        "F1": metrics["f1"],
        "AUROC": metrics["auroc"],
        "KS statistic": complete["ks_statistic"],
        "KS p-value": complete["ks_p_value"],
        "GD minutes": complete["training_seconds"] / 60,
        "Full Retraining minutes": complete["full_retraining_training_seconds"] / 60,
        "Speed-up": complete["speed_up"],
    })

final_gd_results = pd.DataFrame(rows)
display(final_gd_results.round(4))

print(
    "Completed:",
    f"{len(final_gd_results)}/{len(FINAL_SCENARIOS)} final GD scenarios",
)

# How to explain Gradient Difference to a marker

Gradient Difference starts from the trained baseline. Each forgotten row is paired with
one deterministically sampled retained-training row. The method minimises
`L_retain - L_forget`, so it simultaneously learns from retained data and increases loss
on forgotten data. One complete epoch is one complete pass over the forget set. The
trajectory is fixed at five epochs and epoch 5 is reported as the primary result; validation,
test, KS and Full Retraining similarity are not used as an oracle to choose an earlier model.